<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />


# Worksheet 9.5: Attention & Your First GPT — Answers

*Module 9 — Build Your Own LLM.* This is the **answer key** with every cell completed.

The bigram model had one fatal flaw: each token could only see the **single** token before it. To write coherent text, a token needs to look back at **all** the earlier tokens and decide which ones matter. That mechanism is **self-attention** — the single idea that makes transformers (and therefore GPT) work.

In this lab you'll build attention from scratch in small runnable steps, then assemble it into a real (tiny) **GPT** and train it end-to-end.

## 1. Setup

In [8]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

## 2. A first idea: average the past

Suppose each token should become the **average of itself and every token before it** (it must never peek at the future). Here `x` is a tiny sequence: 1 batch, 8 time steps, 2 numbers per step. The slow, obvious way uses a loop.

In [2]:
B, T, C = 1, 8, 2          # batch, time (tokens), channels (numbers per token)
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))      # 'bow' = bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]       # everything up to and including step t
        xbow[b, t] = xprev.mean(0)
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


## 3. The same thing as a matrix multiply

Loops are slow. The trick that makes attention fast: a **lower-triangular** weight matrix whose rows sum to 1 computes that same 'average of the past' in one matrix multiply. Row `t` says how much token `t` draws from each earlier token.

In [3]:
wei = torch.tril(torch.ones(T, T))        # lower-triangular ones
wei = wei / wei.sum(1, keepdim=True)      # make each row sum to 1
xbow2 = wei @ x                           # (T,T) @ (B,T,C) -> (B,T,C)
print("weights (row t = how much token t attends to the past):\n", wei)
print("\nmatches the loop version?", torch.allclose(xbow, xbow2))

weights (row t = how much token t attends to the past):
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

matches the loop version? True


## 4. From averaging to softmax

Real attention doesn't weight the past *equally* — it **learns** the weights. The standard way to turn arbitrary scores into positive weights that sum to 1 is **softmax**. We mask future positions with `-inf` so they get weight 0 after softmax. Right now all the scores are 0, so this still gives equal averaging — but now the weights are *learnable*.

In [4]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float("-inf"))   # block the future
wei = F.softmax(wei, dim=-1)
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


## 5. Self-attention: query, key, value

Now the real thing. Every token emits three vectors:

- **query** — *what am I looking for?*
- **key** — *what do I contain?*
- **value** — *what will I pass on if attended to?*

A token's attention to another = its **query · the other's key**. We scale, apply the causal mask, softmax, then take a weighted sum of **values**.

**TODO:** compute the scaled scores `wei` and apply the causal mask.

In [5]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, head_size)
q = query(x)    # (B, T, head_size)
v = value(x)    # (B, T, head_size)

# attention scores, scaled by 1/sqrt(head_size) to keep them well-behaved
wei = q @ k.transpose(-2, -1) * head_size ** -0.5     # (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))       # causal mask: no peeking ahead
wei = F.softmax(wei, dim=-1)

out = wei @ v   # (B, T, head_size)
print("attention output shape:", out.shape)
print("\nattention weights for the LAST token of sequence 0:")
print(wei[0, -1])

attention output shape: torch.Size([4, 8, 16])

attention weights for the LAST token of sequence 0:
tensor([0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553],
       grad_fn=<SelectBackward0>)


Notice the weights for the last token are **uneven** — the model is choosing which earlier tokens to focus on, instead of averaging them equally. That choice is *learned* during training. The scaling by `1/sqrt(head_size)` keeps the scores from getting so large that softmax becomes a hard, one-hot pick early on.

## 6. Now build a GPT out of it

You have the one idea that matters. Everything left is assembly: run several attention heads in parallel, add a small feed-forward layer, stack the result a few times, and tell the model *where* each token sits.

First, the tokenizer you trained in Lab 9.4 and the hyper-parameters.

In [7]:
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(1337)

# ==========================================================================
#  SCALE-UP KNOB
#  These defaults train in well under a minute on a CPU.
#  Have a GPU? Try the bigger values in the comments for nicer text.
# ==========================================================================
block_size    = 32      # context length         (GPU: 128 or 256)
batch_size    = 16      # sequences per step     (GPU: 64)
n_embd        = 64      # embedding size         (GPU: 256 or 384)
n_head        = 4       # attention heads        (GPU: 6)
n_layer       = 3       # transformer blocks     (GPU: 6)
dropout       = 0.1
max_iters     = 3000    # training steps         (GPU: 5000)
eval_iters    = 100
learning_rate = 3e-3
print("device:", device)

device: mps


## 7. Load the tokenizer and the data

Rather than retraining the BPE tokenizer (that was Lab 9.4's job), we load the merges you saved. Same merges everywhere means token id 42 always means the same thing.

Note that `block_size = 32` now buys us far more context than it would with characters — at roughly 2.3 characters per BPE token, 32 tokens is about 75 characters of text.

In [9]:
# ---- The BPE tokenizer you trained in Lab 9.4. Just run this cell. ----

_merge_list = json.load(open("../data/bpe_merges.json"))
merges = {(a, b): new_id for a, b, new_id in _merge_list}   # in training order

vocab = {i: bytes([i]) for i in range(256)}                 # ids 0-255 are raw bytes
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b]
vocab_size = 256 + len(merges)

def encode(s):
    """Text -> token ids, by applying the learned merges in order."""
    ids = list(s.encode("utf-8"))
    for (a, b), new_id in merges.items():
        out, i, n = [], 0, len(ids)
        while i < n:
            if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                out.append(new_id); i += 2
            else:
                out.append(ids[i]); i += 1
        ids = out
    return ids

def decode(ids):
    """Token ids -> text. errors='replace' because a partly-generated
       multi-byte character is a real possibility when sampling."""
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

NEWLINE = encode("\n")[0]     # handy seed token for generation
print("vocab size:", vocab_size)

vocab size: 768


In [10]:
text = open("../data/tiny_corpus.txt").read()
data = torch.tensor(encode(text), dtype=torch.long)     # takes ~10s
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab:", vocab_size, "| corpus tokens:", len(data))
print("a", block_size, "token window covers ~", round(len(text) / len(data) * block_size), "characters")

vocab: 768 | corpus tokens: 145098
a 32 token window covers ~ 70 characters


## 8. One attention head

This is exactly the attention you built in section 5, wrapped in a module so we can make several. `register_buffer` just stores the causal mask with the model.

In [11]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

## 9. Many heads in parallel

One head learns one kind of relationship. **Multi-head attention** runs several in parallel and concatenates them, so the model can track several things at once (e.g. one head for punctuation, one for subject–verb).

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

## 10. A little per-token 'thinking' layer

After tokens have gathered information via attention, a small **feed-forward** network lets each token process what it collected.

In [13]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 11. The transformer block

A **block** = multi-head attention + feed-forward, each wrapped with a **residual connection** (`x + ...`, which helps gradients flow) and **layer norm** (which keeps the numbers stable). GPT is just a stack of these.

**TODO:** wire up the two residual connections in `forward`.

In [14]:
class Block(nn.Module):
    # Transformer block: tokens first COMMUNICATE (attention), then THINK (feed-forward).
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # attention + residual connection
        x = x + self.ffwd(self.ln2(x))    # feed-forward + residual connection
        return x

## 12. The full GPT

Now we assemble the model:

- a **token** embedding (what is this token?) **+** a **position** embedding (where is it?),
- a stack of `n_layer` transformer blocks,
- a final layer-norm and a linear `lm_head` that produces next-token logits.

`generate` also gains two real-world sampling controls: **temperature** (higher = more random) and **top-k** (only sample from the k most likely tokens).

In [15]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)                       # (B,T,n_embd)
        pos = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                    # (B,T,vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]            # never feed more than block_size
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature    # last step, scaled by temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = GPTLanguageModel().to(device)
print(round(sum(p.numel() for p in model.parameters()) / 1e3, 1), "K parameters")

250.6 K parameters


## 13. Train it

`estimate_loss` averages the loss over several batches so the printed numbers are less noisy, and reports **train** vs **val** loss. Then the familiar loop.

Starting point for a vocabulary of 768 is a loss around `ln(768) ≈ 6.64`; a well-trained model here should land far below that. This takes well under a minute on a CPU.

In [16]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for it in range(max_iters):
    if it % 500 == 0 or it == max_iters - 1:
        l = estimate_loss()
        print(f"step {it:4d} | train loss {l['train']:.3f} | val loss {l['val']:.3f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print("done")

step    0 | train loss 6.810 | val loss 6.810
step  500 | train loss 3.936 | val loss 4.137
step 1000 | train loss 3.625 | val loss 3.904
step 1500 | train loss 3.407 | val loss 3.714
step 2000 | train loss 3.275 | val loss 3.648
step 2500 | train loss 3.189 | val loss 3.552
step 2999 | train loss 3.111 | val loss 3.615
done


## 14. Generate text from YOUR model

Seed with a newline and let the model write. It won't be Shakespeare — but you'll see real words, ALL-CAPS speaker names like the script, and dialogue structure. **You built and trained this.**

In [17]:
context = torch.tensor([[NEWLINE]], dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


other cestimpetious roye bannot sight!
We no name to thees-hear disceptursed, deadved facter.

LADY ANNE:
How reconsent you, from consulty, lebuns of less
Than shall be thy bety double is other now
that a worll.

Second Servingman:
And a thy grace it, an thy leanty and disply; and
Why what into issue begtsicia!
O, do she more thy go mor most him remeptored,--
Hastt them by me!

MENENIUS:
Ipholy foo thy bant thy son your inal: how act.

RIVERS:
I'll being not jourts I do fuldier,
These lip me to ch this this commons himselves. Yeece bearlusy.
We him more the hithilt a how all unface.

Second Servingman:
Why, how of me follow my fook he with.
ThereT


## 15. Temperature & top-k: controlling creativity

Real LLMs expose these same knobs:

- **temperature** < 1 → safer, more repetitive; > 1 → wilder, more typos.
- **top-k** → only ever sample from the *k* most likely next tokens (cuts off nonsense).

**Try it:** change the temperatures or `top_k` and re-run.

In [18]:
for temp in [0.5, 1.0, 1.5]:
    print(f"\n===== temperature = {temp} (top_k=20) =====")
    out = model.generate(context, max_new_tokens=150, temperature=temp, top_k=20)
    print(decode(out[0].tolist()))


===== temperature = 0.5 (top_k=20) =====

I'll broke theep the wond, so.

MENENIUS:
Yet I what is done; you what you well yourselves
The wars of his chase reds: he will to tree of honour
That time with his please
Than he well him fears!

BRUTUS:
I thank, say, as I do not banish'd,
And banisher, you were you hear him what herse that how
The send of our lord of himself.



===== temperature = 1.0 (top_k=20) =====

Or live these reparct of youseld,
And pursured, what he show must were in himself:
Bet, for my follow saken you will love the wom of her,
It shall makes beforeth cot with him
If, being being wot your heart therebery hithees wo,
That show to belly, for how false; and, my part.

MENENIUS:
Yet I am brieak;
Yet,
That son that honours him than to

===== temperature = 1.5 (top_k=20) =====

Art, to be dellow to me thye and the kisice is my sle;
Oft you would have hear tedity, if taked: blooy
And not sigh.

GLOUCESTER:
Arf, madam.

MENENIUS:
The wreast, I could comtecting aid
Yet you 

## Recap — you built a GPT

Your mini-GPT contains every essential ingredient of a real large language model:

| Piece | What it does | Where you met it |
|---|---|---|
| BPE tokenizer | text ↔ token ids, learned from data | Lab 9.4 |
| token + position embeddings | what each token is, and where it sits | Labs 9.4, this lab |
| multi-head self-attention | tokens look back and decide what matters | this lab |
| feed-forward + residual + layer norm | per-token thinking, stable training | this lab |
| cross-entropy + AdamW loop | learning by predicting the next token | Lab 9.3 |
| temperature / top-k sampling | controlling how text is generated | this lab |

The difference between this and ChatGPT is **scale** (billions of parameters, trillions of tokens, big GPUs) and some extra training stages — **not** the core idea. You now understand that core idea end-to-end.

Next: **Worksheet 9.6 — Fine-Tuning & Poisoning**, where you adapt this model to security text and then attack it.